# 📘 Notebook 01 – Data Inspection
**Project:** SentimentIQ – AI-Powered Sentiment Analysis Web Application

**Goal of this notebook:** get to know the raw `Tweets.csv` file *before* we touch it: its size, columns, data types, missing values, duplicates and sentiment labels.

**Why this matters:** every ML project starts with understanding the data. Problems found here (missing values, duplicates, wrong labels) are cheaper to fix now than after training a model.

## 1. Import Required Libraries

In [ ]:
# Import the libraries used for data inspection
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")                  # keep the output clean
pd.set_option("display.max_columns", None)         # show every column
pd.set_option("display.max_colwidth", 120)         # show longer text values

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

## 2. Set Project Paths and Load `Tweets.csv`

In [ ]:
# Set up project paths (works on your D: drive path, or falls back to the parent of 02_Notebooks)
from pathlib import Path

BASE_DIR = Path(r"D:\DataScience\notebooks\SentimentIQ")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd().parent          # fallback: notebook is running inside 02_Notebooks/

RAW_PATH       = BASE_DIR / "01_Data" / "Raw" / "Tweets.csv"
PROCESSED_DIR  = BASE_DIR / "01_Data" / "Processed"
CLEANED_PATH   = PROCESSED_DIR / "cleaned_tweets.csv"
MODELS_DIR     = BASE_DIR / "03_Models"
REPORTS_DIR    = BASE_DIR / "05_Reports"

for folder in (PROCESSED_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42      # same seed everywhere -> reproducible results
TEST_SIZE    = 0.20    # 80% training / 20% testing

print("Project folder :", BASE_DIR)
print("Raw data exists:", RAW_PATH.exists())

In [ ]:
# Load Tweets.csv into a DataFrame (with friendly error handling)
try:
    df = pd.read_csv(RAW_PATH)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Tweets.csv was not found at: {RAW_PATH}\n"
        "Copy the file into 01_Data/Raw/ and run this cell again."
    )
except UnicodeDecodeError:
    df = pd.read_csv(RAW_PATH, encoding="latin-1")   # fallback encoding

print("✅ Dataset loaded successfully")

## 3. First Five Records

In [ ]:
# Display the first five rows to see what the data looks like
df.head()

## 4. Last Five Records

In [ ]:
# Display the last five rows (useful to confirm the file was fully loaded)
df.tail()

## 5. Dataset Dimensions

In [ ]:
# Check number of rows (tweets) and columns (features)
rows, cols = df.shape
print(f"Number of rows    : {rows:,}")
print(f"Number of columns : {cols}")

**Expected output (original Kaggle file):** `14,640` rows and `15` columns.

## 6. Column Names

In [ ]:
# List all column names
for i, name in enumerate(df.columns, start=1):
    print(f"{i:>2}. {name}")

## 7. Data Types

In [ ]:
# Check the data type of each column (object = text, int64/float64 = numbers)
df.dtypes.to_frame("data_type")

## 8. Dataset Information

In [ ]:
# Concise summary: non-null counts, dtypes and memory usage
df.info()

## 9. Missing Value Analysis

In [ ]:
# Count missing values per column and show them as a percentage
missing = pd.DataFrame({
    "missing_count":   df.isnull().sum(),
    "missing_percent": (df.isnull().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

display(missing[missing["missing_count"] > 0])

In [ ]:
# Check the two columns that matter most for modelling: text (input) and airline_sentiment (target)
print("Missing in 'text'              :", df["text"].isnull().sum())
print("Missing in 'airline_sentiment' :", df["airline_sentiment"].isnull().sum())
print("Blank text values              :", (df["text"].astype(str).str.strip() == "").sum())

**Observation:** columns such as `airline_sentiment_gold`, `negativereason_gold` and `tweet_coord` are almost entirely empty, and `negativereason` is empty for every non-negative tweet. Our two modelling columns (`text`, `airline_sentiment`) are complete.

## 10. Duplicate Record Analysis

In [ ]:
# Check three kinds of duplicates: identical rows, repeated tweet IDs and repeated tweet text
full_dups = df.duplicated().sum()
id_dups   = df["tweet_id"].duplicated().sum()
text_dups = df["text"].duplicated().sum()

print(f"Fully identical rows : {full_dups}")
print(f"Repeated tweet_id    : {id_dups}")
print(f"Repeated tweet text  : {text_dups}")

In [ ]:
# Look at a few repeated tweets (retweets and copy-pasted text are common on Twitter)
repeated = df[df["text"].duplicated(keep=False)].sort_values("text")
repeated[["airline_sentiment", "airline", "text"]].head(10)

**Why it matters:** if the same tweet appears in both the training and test sets, the model is tested on something it has already seen, which inflates accuracy. We will remove duplicates in Notebook 03.

## 11. Unique Sentiment Categories

In [ ]:
# Find the unique values of the target column and how often each occurs
print("Unique sentiment labels:", df["airline_sentiment"].dropna().unique().tolist())
print()
print(df["airline_sentiment"].value_counts())

**Expected output:** `negative` ≈ 9,178 · `neutral` ≈ 3,099 · `positive` ≈ 2,363 – the data is **imbalanced** (mostly negative).

## 12. Basic Dataset Summary

In [ ]:
# Statistical summary of the columns we care about
df[["airline_sentiment", "airline_sentiment_confidence", "airline", "text"]].describe(include="all")

In [ ]:
# Build a compact one-page summary of the dataset
created = pd.to_datetime(df["tweet_created"], errors="coerce", utc=True)

summary = {
    "Total tweets":           f"{len(df):,}",
    "Total columns":          df.shape[1],
    "Airlines covered":       df["airline"].nunique(),
    "Sentiment classes":      df["airline_sentiment"].nunique(),
    "Date range":             f"{created.min().date()}  to  {created.max().date()}",
    "Average tweet length":   f"{df['text'].str.len().mean():.1f} characters",
    "Most common sentiment":  df["airline_sentiment"].mode()[0],
}
pd.Series(summary, name="Value").to_frame()

## ✅ Key Findings
- The dataset has **≈14.6k tweets** about **6 US airlines**.
- The target `airline_sentiment` has 3 classes and **no missing values**; the class distribution is imbalanced.
- A small number of duplicate tweets exist → will be removed.
- Many columns are mostly empty or would leak the answer (`negativereason`, confidence scores) → will not be used for modelling.

**Next:** Notebook 02 – Exploratory Data Analysis.